In [1]:
!nvidia-smi
print("GPU 상태 확인 완료!")

Mon Jul 27 00:34:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =====================================================================
# 1단계: YOLO 위치 검출 모델 학습 (Google Colab에서 실행 권장)
# =====================================================================
# 사용 순서
# 1) https://colab.research.google.com 접속 → 새 노트북
# 2) 상단 메뉴 [런타임] > [런타임 유형 변경] > 하드웨어 가속기 = GPU 선택
# 3) 이 파일 내용을 셀에 붙여넣고 위에서부터 순서대로 실행
# 4) ROBOFLOW_API_KEY, WORKSPACE, PROJECT, VERSION 은 본인 Roboflow
#    프로젝트 페이지 우측 상단 "Download Dataset" 버튼 눌렀을 때 나오는
#    코드에서 그대로 복사하면 됩니다.
# =====================================================================

# --- 설치 ---
!pip install ultralytics roboflow -q
# %pip install ultralytics roboflow -q

# --- 1) Roboflow에서 바운딩박스 라벨 포함 데이터셋 다운로드 ---
from roboflow import Roboflow

ROBOFLOW_API_KEY = "08pEqA03ywLShGQ8Vk09"
WORKSPACE = "s-workspace-ntur3"
PROJECT = "trash_line_3class"
VERSION = 1  # Roboflow 프로젝트 버전 번호

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)

# 사용 가능한 버전 번호 확인 (VERSION 값이 실제 존재하는지 먼저 체크)
print("사용 가능한 버전:", [v.version for v in project.versions()])

dataset = project.version(VERSION).download("yolov8")
# 다운로드된 폴더 안에 data.yaml (클래스 이름 정의) + train/valid/test 가 생김

print("데이터셋 위치:", dataset.location)

# --- 2) YOLOv8n(nano)으로 전이학습 ---
# nano 버전을 쓰는 이유: Jetson Nano처럼 연산이 약한 보드에 올리기엔
# 가장 가벼운 버전이 안전합니다. (s/m/l/x 로 갈수록 무겁고 정확하지만 느려짐)
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    name="waste_yolo",
    patience=20,       # 20 epoch 동안 성능 개선 없으면 조기 종료
)

# --- 3) 학습 결과 확인 ---
# 학습이 끝나면 다음 경로에 결과가 저장됩니다.
#   runs/detect/waste_yolo/weights/best.pt   <- 최종 모델 (이걸 사용)
#   runs/detect/waste_yolo/confusion_matrix.png  <- 클래스별 오분류 확인
#   runs/detect/waste_yolo/results.png           <- 학습 곡선(mAP, loss 등)
#
# best.pt 를 다운로드해서 로컬에 저장해두세요.
# (Colab 왼쪽 파일 탐색기에서 우클릭 > 다운로드)

# --- 4) 학습된 모델로 실제 이미지 테스트 (선택) ---
# best_model = YOLO("runs/detect/waste_yolo/weights/best.pt")
# results = best_model.predict("테스트할_이미지_경로.jpg", save=True, conf=0.5)

# =====================================================================
# 다음 단계: best.pt 로 원본 학습 이미지들의 바운딩박스를 잘라내서
# CNN 분류기용 데이터셋을 만듭니다. -> 2_crop_bboxes_for_cnn.py 참고
# =====================================================================

loading Roboflow workspace...
loading Roboflow project...
사용 가능한 버전: ['1']
데이터셋 위치: D:\PROJECT\AI\Jetson_Recycling-conveyor-belt\trash_line_3class\trash_line_3class-1
New https://pypi.org/project/ultralytics/8.4.108 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.0  Python-3.8.10 torch-2.4.1+cpu CPU (Intel Core(TM) i7-9700 3.00GHz)
engine\trainer: task=detect, mode=train, model=yolov8n.pt, data=D:\PROJECT\AI\Jetson_Recycling-conveyor-belt\trash_line_3class\trash_line_3class-1/data.yaml, epochs=100, time=None, patience=20, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=waste_yolo, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=Fal

train: Scanning D:\PROJECT\AI\Jetson_Recycling-conveyor-belt\trash_line_3class\trash_line_3class-1\train\labels... 2472 images, 0 backgrounds, 0 corrupt: 100%|██████████| 2472/2472 [00:05<00:00, 419.99it/s]

train: WARNING  D:\PROJECT\AI\Jetson_Recycling-conveyor-belt\trash_line_3class\trash_line_3class-1\train\images\paper510_jpg.rf.bd4642e2e9476e5dec0157032b2d9df2.jpg: 1 duplicate labels removed
train: WARNING  D:\PROJECT\AI\Jetson_Recycling-conveyor-belt\trash_line_3class\trash_line_3class-1\train\images\paper537_jpg.rf.10d2899214630c7d0dc033346014f8d1.jpg: 1 duplicate labels removed


train: New cache created: D:\PROJECT\AI\Jetson_Recycling-conveyor-belt\trash_line_3class\trash_line_3class-1\train\labels.cache


val: Scanning D:\PROJECT\AI\Jetson_Recycling-conveyor-belt\trash_line_3class\trash_line_3class-1\valid\labels... 701 images, 0 backgrounds, 0 corrupt: 100%|██████████| 701/701 [00:00<00:00, 1403.24it/s]

val: WARNING  D:\PROJECT\AI\Jetson_Recycling-conveyor-belt\trash_line_3class\trash_line_3class-1\valid\images\paper_0316_jpg.rf.4f7eed90d008b1d945bc8329bf2499ad.jpg: 1 duplicate labels removed
val: New cache created: D:\PROJECT\AI\Jetson_Recycling-conveyor-belt\trash_line_3class\trash_line_3class-1\valid\labels.cache
Plotting labels to runs\detect\waste_yolo\labels.jpg... 


optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.001429, momentum=0.9) with parameter groups 63 weight(decay=0.0), 70 weight(decay=0.0005), 69 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to runs\detect\waste_yolo
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      1/100         0G     0.6236      3.045      1.131         44        640:  15%|█▍        | 23/155 [01:18<07:40,  3.49s/it]

In [ ]:
# # --- 3-1) best.pt를 Google Drive에 백업 (런타임 끊겨도 안전하게 보관) ---
# from google.colab import drive
# drive.mount('/content/drive')

# import shutil, os

# SAVE_DIR = "/content/drive/MyDrive/waste_yolo_results"
# os.makedirs(SAVE_DIR, exist_ok=True)

# shutil.copy("/content/runs/detect/waste_yolo/weights/best.pt", f"{SAVE_DIR}/best.pt")
# shutil.copy("/content/runs/detect/waste_yolo/weights/last.pt", f"{SAVE_DIR}/last.pt")

# print(f"저장 완료: {SAVE_DIR}/best.pt")


#===========================================================================
# # --- 3-1) 결과 폴더를 통째로 압축해서 다운로드 ---
# # 로컬에서 돌렸을 때와 똑같이 first_test/runs/detect/waste_yolo/ 구조가 되도록,
# # 압축 파일을 풀면 그대로 first_test/ 밑에 넣을 수 있는 형태로 만듭니다.
# import shutil

# shutil.make_archive("/content/runs", "zip", "/content", "runs")

# from google.colab import files
# files.download("/content/runs.zip")

# print("runs.zip 다운로드 완료.")
# print("압축 풀어서 나온 runs 폴더를 first_test/ 안에 그대로 넣으면")
# print("로컬에서 돌린 것과 동일하게 first_test/runs/detect/waste_yolo/... 경로가 됩니다.")


In [ ]:
# --- 3-1) Google Drive에 저장 + 자동 다운로드를 위한 공유 설정 ---
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

SAVE_DIR = "/content/drive/MyDrive/waste_yolo_results"
os.makedirs(SAVE_DIR, exist_ok=True)
shutil.copy("/content/runs/detect/waste_yolo/weights/best.pt", f"{SAVE_DIR}/best.pt")
shutil.copy("/content/runs/detect/waste_yolo/weights/last.pt", f"{SAVE_DIR}/last.pt")

# 폴더를 "링크가 있는 사람은 보기 가능"으로 공유 설정 (로컬 스크립트가 인증 없이 받아갈 수 있도록)
from google.colab import auth
auth.authenticate_user()

from googleapiclient.discovery import build
drive_service = build('drive', 'v3')

result = drive_service.files().list(
    q="name='waste_yolo_results' and mimeType='application/vnd.google-apps.folder'",
    spaces='drive', fields='files(id, name)'
).execute()
folder_id = result['files'][0]['id']

drive_service.permissions().create(
    fileId=folder_id,
    body={'type': 'anyone', 'role': 'reader'},
).execute()

print("저장 완료:", SAVE_DIR)
print("폴더 ID (한 번만 복사해서 first_test/pull_results.py의 FOLDER_ID에 붙여넣으세요):")
print(folder_id)

저장 완료: /content/drive/MyDrive/waste_yolo_results
폴더 ID (한 번만 복사해서 first_test/pull_results.py의 FOLDER_ID에 붙여넣으세요):
1NVM42UWV7zxyu_MisV7o4fllAnvL7d6a
